<a href="https://colab.research.google.com/github/Irvingrer/Optimizador-de-distancias-log-sticas-OSMR-y-geod-sica-/blob/main/OSMR_matpre.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
# Importar todas las librerías necesarias al principio
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import networkx as nx
import requests
import openpyxl
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
import os
from datetime import datetime, timedelta # Importar timedelta
import time
from google.colab import drive, files # Import drive and files for Colab
import shutil # Import shutil for file operations

# --- CONFIGURACIÓN MANUAL DE PARÁMETROS ---
# Modifica las siguientes variables para controlar el procesamiento:

# Lista de códigos de METs a procesar. Ejemplo: ['MET Celaya', 'MET Querétaro']
# Asegúrate de que los códigos coincidan exactamente con los de tu archivo Excel.
mets_seleccionados_manual = ['MET CDMX2','MET Ceylán'] # <<< MODIFICA AQUÍ TUS MET(s) SELECCIONADOS

# Lista de números de Ruta a procesar. Ejemplo: [1, 4, 6]
# Asegúrate de que los números de ruta coincidan exactamente con los de tu archivo Excel.
# Esta lista solo se usará si todas_rutas_manual es False.
rutas_seleccionadas_manual = [19,20,21,24,25,26,30] # <<< MODIFICA AQUÍ TUS RUTAS SELECCIONADAS (si no procesas todas)

# Si es True, procesará el 100% de las rutas disponibles,
# ignorando el valor de rutas_seleccionadas_manual.
# Si es False, usará el valor de rutas_seleccionadas_manual.
todas_rutas_manual = True # <<< PON TRUE PARA PROCESAR TODAS LAS RUTAS, FALSE PARA USAR rutas_seleccionadas_manual

# Si es True, procesará el 100% de los clientes en las rutas seleccionadas.
# Si es False, usará el valor de num_clientes_manual para limitar la cantidad de clientes.
todos_clientes_manual = True # <<< PON TRUE PARA PROCESAR TODOS LOS CLIENTES, FALSE PARA LIMITAR POR num_clientes_manual

# Número máximo de clientes a procesar por ruta (incluyendo el MET).
# Solo se usará if todos_clientes_manual es False.
num_clientes_manual = 10 # <<< MODIFICA AQUÍ EL NÚMERO MÁXIMO DE CLIENTES POR RUTA (INCLUYENDO MET)

# Hora de inicio fija para las rutas (en formato HH:MM).
hora_inicio_ruta = "07:00" # <<< MODIFICA AQUÍ LA HORA DE INICIO DE LAS RUTAS

# --- FIN DE LA CONFIGURACIÓN MANUAL ---


# 1. Montar Google Drive (necesario para acceder al archivo de entrada if está allí)
# Si ya lo has montado en una celda anterior, esta línea no hará daño.
# print("Montando Google Drive...")
# try:
#     drive.mount('/content/drive')
#     print("Google Drive montado exitosamente.")
# except Exception as e:
#     print(f"Error montando Google Drive: {e}. Asegúrate de haber permitido el acceso.")

# 2. Asegurar la accesibilidad del archivo de entrada
# Copiar el archivo desde Google Drive al almacenamiento temporal de Colab
# **CRÍTICO:** Verifica la ruta exacta de tu archivo 'Clientes_Ruta.xlsx' en Google Drive.
# Haz clic derecho sobre el archivo en el explorador de archivos de Google Drive dentro de Colab y selecciona "Copiar ruta".
# Reemplaza la ruta del marcador de posición a continuación con la ruta copiada.
# google_drive_file_path = "/content/drive/MyDrive/Colab Notebooks/Archivos/Matriz precalculada Celaya2.xlsx" # <<< VERIFICA ESTA RUTA

# ARCHIVO_DATOS = '/tmp/Matriz precalculada Ceylan.xlsx' # Ruta temporal en Colab

# print(f"Intentando copiar el archivo desde '{google_drive_file_path}' a '{ARCHIVO_DATOS}'...")
# try:
#     # Usamos shutil.copy instead of !cp for better error handling within Python
#     shutil.copy(google_drive_file_path, ARCHIVO_DATOS)
#     print(f"Archivo copiado exitosamente a {ARCHIVO_DATOS}")
# except FileNotFoundError:
#      print(f"Error: El archivo '{google_drive_file_path}' no fue encontrado en Google Drive. Por favor, verifica la ruta.")
#      # Re-raise the error to stop execution if the file is not found
#      raise
# except Exception as e:
#     print(f"Error al copiar el archivo desde Google Drive. Por favor, verifica que el archivo exista en '{google_drive_file_path}' y que Google Drive esté montado correctamente. Error: {e}")
#     # Si la copia falla, pd.read_excel lanzará un error si el archivo no está en /tmp
#     # Re-raise other exceptions during file copy
#     raise

# 2. Cargar archivo desde la máquina local
print("Por favor, selecciona el archivo Excel de entrada.")
uploaded = files.upload()

if not uploaded:
    print("No se seleccionó ningún archivo. Cancelando ejecución.")
    raise FileNotFoundError("No se seleccionó ningún archivo de entrada.")

# Asumir que el usuario subirá un solo archivo
ARCHIVO_DATOS = next(iter(uploaded))
print(f"Archivo '{ARCHIVO_DATOS}' cargado exitosamente.")


# 3. Cargar datos desde Excel
try:
    print(f"Intentando cargar datos desde {ARCHIVO_DATOS}...")
    # Usamos engine='openpyxl' explícitamente ya que ya importamos la librería
    # Cargar la primera hoja, asumiendo que contiene todos los puntos con la secuencia
    df_puntos_secuencia = pd.read_excel(ARCHIVO_DATOS, sheet_name=0, engine='openpyxl')
    print("Datos cargados exitosamente.")
    print(f"Dimensiones de df_puntos_secuencia: {df_puntos_secuencia.shape}")

    # Verificar que las columnas necesarias existan
    # Actualizado para que coincida con los nombres de columna reales proporcionados por el usuario
    required_cols = ['🏠 MET', '🛣️ Ruta', '🔢 Secuencia', 'Tipo', 'Codigo', 'Longitud', 'Latitud', '📚 Nombre']
    if not all(col in df_puntos_secuencia.columns for col in required_cols):
        missing = [col for col in required_cols if col not in df_puntos_secuencia.columns]
        print(f"Error: Faltan columnas requeridas en el archivo Excel: {missing}")
        raise ValueError("Faltan columnas requeridas en el archivo Excel.")

except FileNotFoundError:
    print(f"Error: El archivo {ARCHIVO_DATOS} no fue encontrado. Por favor, asegúrate de que la ruta en Google Drive sea correcta y que el archivo exista.")
    raise # Re-lanzar el error para detener la ejecución if el archivo no se encuentra.
except Exception as e:
    print(f"Ocurrió un error al cargar los datos: {e}")
    raise # Re-lanzar otras excepciones durante la carga de datos.

# 4. Asignar parámetros de procesamiento desde la configuración manual
# Se usan las variables definidas al principio del script.
print("Asignando parámetros desde la configuración manual...")
mets_seleccionados = mets_seleccionados_manual

# Usar todas las rutas disponibles if todas_rutas_manual es True, de lo contrario usar rutas_seleccionadas_manual
if todas_rutas_manual and 'df_puntos_secuencia' in locals() and not df_puntos_secuencia.empty:
    # Convertir la columna 'Ruta' a tipo string antes de usar .dropna().unique()
    # Esto evita problemas if hay valores numéricos y NaN
    rutas_disponibles = sorted(df_puntos_secuencia['🛣️ Ruta'].astype(str).dropna().unique())
    rutas_seleccionadas = rutas_disponibles
    print("Procesando todas las rutas disponibles.")
elif 'df_puntos_secuencia' in locals() and not df_puntos_secuencia.empty:
    # Convertir la columna 'Ruta' a tipo string antes de usar .dropna().unique() y la lista manual a string
    rutas_disponibles = sorted(df_puntos_secuencia['🛣️ Ruta'].astype(str).dropna().unique())
    rutas_seleccionadas_manual_str = [str(ruta) for ruta in rutas_seleccionadas_manual]
    # Validar que las selecciones manuales existan en el dataframe cargado (como strings)
    invalid_rutas = [ruta for ruta in rutas_seleccionadas_manual_str if ruta not in rutas_disponibles]
    if invalid_rutas:
        print(f"Advertencia: Las siguientes Rutas seleccionadas manualmente no se encontraron en los datos: {invalid_rutas}. Serán ignoradas.")
    rutas_seleccionadas = [ruta for ruta in rutas_seleccionadas_manual_str if ruta in rutas_disponibles]
    # Convertir las rutas seleccionadas de nuevo a int if posible, para mantener el tipo original
    rutas_seleccionadas = [int(ruta) if ruta.isdigit() else ruta for ruta in rutas_seleccionadas]
    print("Procesando rutas seleccionadas manualmente.")
else:
     rutas_seleccionadas = [] # Asegurarse de que la lista esté vacía if df_puntos_secuencia no está disponible
     print("Advertencia: No se pudo determinar las rutas a procesar (DataFrame no cargado o vacío).")

# Validar que los MET(s) seleccionados existan en el dataframe cargado
if 'df_puntos_secuencia' in locals() and not df_puntos_secuencia.empty:
    mets_disponibles = sorted(df_puntos_secuencia['🏠 MET'].dropna().unique())
    invalid_mets = [met for met in mets_seleccionados if met not in mets_disponibles]
    if invalid_mets:
        print(f"Advertencia: Los siguientes MET(s) seleccionados manualmente no se encontraron en los datos: {invalid_mets}. Serán ignorados.")
    mets_seleccionados = [met for met in mets_seleccionados if met in mets_disponibles]
else:
    print("Error: El DataFrame de puntos de secuencia no se cargó o está vacío. No se pueden validar los MET(s) seleccionados.")
    mets_seleccionados = [] # Asegurarse de que la lista esté vacía if df_puntos_secuencia no está disponible

# Configurar el número de clientes a procesar por ruta
num_clientes_a_procesar = float('inf') # Por defecto, procesar todos
if not todos_clientes_manual:
    num_clientes_a_procesar = num_clientes_manual
    print(f"Limitando el procesamiento a los primeros {num_clientes_a_procesar} clientes por ruta.")
else:
    print("Procesando todos los clientes en las rutas seleccionadas.")

# Convertir la hora de inicio a objeto datetime para cálculos
try:
    hora_inicio_dt = datetime.strptime(hora_inicio_ruta, '%H:%M').time()
except ValueError:
    print(f"Error: El formato de hora de inicio '{hora_inicio_ruta}' no es válido. Debe ser HH:MM. Usando 07:00 por defecto.")
    hora_inicio_dt = datetime.strptime("07:00", '%H:%M').time()


# Mostrar los parámetros de procesamiento
print(f'Procesando con MET(s): {mets_seleccionados}')
if todas_rutas_manual:
    print('Procesando TODAS las rutas disponibles.')
else:
    print(f'Procesando con Ruta(s): {rutas_seleccionadas}')
print(f'Hora de inicio de rutas: {hora_inicio_dt.strftime("%H:%M")}')


# 5. Obtener distancia y tiempo entre dos puntos con OSRM
def obtener_distancia_tiempo_osrm(punto1, punto2):
    url_base = 'http://router.project-osrm.org/route/v1/driving/'
    session = requests.Session()  # Usar sesión para acelerar múltiples requests

    # Añadir un pequeño retraso para evitar saturar el servidor OSRM
    time.sleep(0.1)
    # Asegurarse de que las coordenadas sean válidas (no NaN o None)
    if pd.isna(punto1[0]) or pd.isna(punto1[1]) or pd.isna(punto2[0]) or pd.isna(punto2[1]):
        print(f"Advertencia: Coordenadas inválidas para los puntos {punto1} y {punto2}. Estableciendo distancia y tiempo a infinito.")
        return float('inf'), float('inf')

    coords = f"{punto1[0]},{punto1[1]};{punto2[0]},{punto2[1]}"
    url = url_base + coords + '?overview=false'
    try:
        r = session.get(url, timeout=30) # Aumentado el timeout para las solicitudes OSRM de 15 a 30 segundos
        r.raise_for_status() # Lanzar excepción para códigos de estado de error HTTP
        data = r.json()
        # Asegurarse de que haya rutas y al menos una ruta con distancia y duración válidas
        if 'routes' in data and data['routes'] and 'distance' in data['routes'][0] and 'duration' in data['routes'][0]:
             distance_km = data['routes'][0]['distance']/1000
             duration_seconds = data['routes'][0]['duration']
             return distance_km, duration_seconds
        else:
             print(f"Advertencia: No se encontró una ruta válida con distancia y duración para los puntos {punto1} y {punto2}. Datos de respuesta: {data}. Estableciendo distancia y tiempo a infinito.")
             return float('inf'), float('inf') # Establecer distancia y tiempo a infinito if no hay ruta válida con ambos datos
    except requests.exceptions.RequestException as e:
        print(f"Advertencia: Falló la solicitud a OSRM para los puntos {punto1} y {punto2}: {e}. Estableciendo distancia y tiempo a infinito.")
        return float('inf'), float('inf')
    except Exception as e:
         print(f"Advertencia: Ocurrió un error inesperado durante la solicitud OSRM para los puntos {punto1} y {punto2}: {e}. Estableciendo distancia y tiempo a infinito.")
         return float('inf'), float('inf')


# Procesar parámetros seleccionados y preparar datos para Excel
print("Iniciando el procesamiento de rutas por secuencia...")
output_dir = '/tmp/outputs' # Directorio de salida en Colab
print(f"Asegurando que el directorio de salida '{output_dir}' exista...")
os.makedirs(output_dir, exist_ok=True) # Asegurar que el directorio exista
fecha_hora = datetime.now().strftime('%Y%m%d_%H%M%S')

export_rows = []
resumen_rutas = []
total_clientes_general = 0
distancia_total_general = 0
tiempo_total_general_segundos = 0
max_dist_general = 0
max_from_general = ''
max_to_general = ''
max_time_general_segundos = 0
max_time_from_general = ''
max_time_to_general = ''


# Verificar que las listas de selección no estén vacías antes de iterar
if not mets_seleccionados:
    print("Advertencia: No se han seleccionado MET(s). No se procesarán rutas.")
elif not rutas_seleccionadas:
     print("Advertencia: No se han seleccionado rutas. No se procesarán rutas.")
elif 'df_puntos_secuencia' not in locals() or df_puntos_secuencia.empty:
     print("Error: El DataFrame de puntos de secuencia no se cargó correctamente o está vacío. No se puede proceder.")
else:
    for met in mets_seleccionados:
        for ruta in rutas_seleccionadas:
            print(f"Procesando MET: {met}, Ruta: {ruta}")
            # Filtrar y ordenar los puntos por Secuencia
            puntos_ruta_df_filtered = df_puntos_secuencia[(df_puntos_secuencia['🏠 MET'] == met) & (df_puntos_secuencia['🛣️ Ruta'].astype(str) == str(ruta))].sort_values(by='🔢 Secuencia')


            # Apply head only if not processing all clients
            if not todos_clientes_manual:
                 puntos_ruta_df = puntos_ruta_df_filtered.head(num_clientes_a_procesar).reset_index(drop=True)
            else:
                 puntos_ruta_df = puntos_ruta_df_filtered.reset_index(drop=True)


            if puntos_ruta_df.empty:
                print(f"Advertencia: No se encontraron puntos para MET {met}, Ruta {ruta} o no se seleccionaron puntos. Omitiendo.")
                continue

            puntos = [(row['Longitud'], row['Latitud']) for _, row in puntos_ruta_df.iterrows()] # Corrected column names
            codigos = list(puntos_ruta_df['Codigo']) # Corrected column name
            nombres = list(puntos_ruta_df['📚 Nombre'])
            tipos = list(puntos_ruta_df['Tipo']) # Use the 'Tipo' column from the Excel

            if len(puntos) < 2:
                 print(f"Advertencia: No hay suficientes puntos ({len(puntos)}) para calcular distancias y tiempos para MET {met}, Ruta {ruta}. Se necesitan al menos 2 puntos. Omitiendo.")
                 continue

            distancias = []
            tiempos_segundos = []
            recorrido_codigos = codigos
            distancia_total_ruta = 0
            tiempo_total_ruta_segundos = 0
            max_dist = 0
            max_from = ''
            max_to = ''
            max_time_segundos = 0
            max_time_from = ''
            max_time_to = ''


            print(f"Calculando distancias y tiempos consecutivos para {len(puntos)} puntos...")
            for i in range(len(puntos) - 1):
                 punto_actual = puntos[i]
                 punto_siguiente = puntos[i+1]
                 dist, time_sec = obtener_distancia_tiempo_osrm(punto_actual, punto_siguiente)
                 distancias.append(dist)
                 tiempos_segundos.append(time_sec)

                 if dist != float('inf'):
                     distancia_total_ruta += dist
                     if dist > max_dist:
                         max_dist = dist
                         max_from = codigos[i]
                         max_to = codigos[i+1]

                 if time_sec != float('inf'):
                     tiempo_total_ruta_segundos += time_sec
                     if time_sec > max_time_segundos:
                         max_time_segundos = time_sec
                         max_time_from = codigos[i]
                         max_time_to = codigos[i+1]
                 else:
                      # Manejar el caso donde la distancia o el tiempo es infinito
                      if max_dist == 0: # If aún no se ha encontrado una distancia finita máxima
                          max_dist = float('inf')
                          max_from = codigos[i]
                          max_to = codigos[i+1]
                      if max_time_segundos == 0: # If aún no se ha encontrado un tiempo finito máximo
                          max_time_segundos = float('inf')
                          max_time_from = codigos[i]
                          max_time_to = codigos[i+1]


            total_clientes_ruta = len(puntos) - 1 # Restar el MET
            distancia_promedio_ruta = distancia_total_ruta / total_clientes_ruta if total_clientes_ruta > 0 and distancia_total_ruta != float('inf') else float('inf')
            tiempo_promedio_ruta_segundos = tiempo_total_ruta_segundos / total_clientes_ruta if total_clientes_ruta > 0 and tiempo_total_ruta_segundos != float('inf') else float('inf')


            resumen_rutas.append({
                'MET': met,
                'Ruta': ruta,
                'Clientes_en_ruta': total_clientes_ruta, # Número de clientes en la ruta (excluyendo MET)
                'Distancia_total_km': distancia_total_ruta,
                'Distancia_promedio_cliente_km': distancia_promedio_ruta,
                'Distancia_maxima_km': max_dist,
                'De_Distancia': max_from,
                'A_Distancia': max_to,
                'Tiempo_total_segundos': tiempo_total_ruta_segundos,
                'Tiempo_total_HHMMSS': str(timedelta(seconds=tiempo_total_ruta_segundos)) if tiempo_total_ruta_segundos != float('inf') else float('inf'),
                'Tiempo_promedio_cliente_segundos': tiempo_promedio_ruta_segundos,
                'Tiempo_promedio_cliente_HHMMSS': str(timedelta(seconds=tiempo_promedio_ruta_segundos)) if tiempo_promedio_ruta_segundos != float('inf') else float('inf'),
                'Tiempo_maximo_segundos': max_time_segundos,
                'Tiempo_maximo_HHMMSS': str(timedelta(seconds=max_time_segundos)) if max_time_segundos != float('inf') else float('inf'),
                'De_Tiempo': max_time_from,
                'A_Tiempo': max_time_to
            })

            if distancia_total_ruta != float('inf'):
                total_clientes_general += total_clientes_ruta
                distancia_total_general += distancia_total_ruta
                # Actualizar max_dist_general only if max_dist es finito
                if max_dist != float('inf'):
                    if max_dist_general == 0 or (max_dist != float('inf') and max_dist > max_dist_general):
                        max_dist_general = max_dist
                        max_from_general = max_from
                        max_to_general = max_to
                elif max_dist_general == 0: # If this is the first route and max_dist is inf
                     max_dist_general = float('inf')
                     max_from_general = max_from
                     max_to_general = max_to

            if tiempo_total_ruta_segundos != float('inf'):
                 tiempo_total_general_segundos += tiempo_total_ruta_segundos
                 # Actualizar max_time_general only if max_time es finito
                 if max_time_segundos != float('inf'):
                      if max_time_general_segundos == 0 or (max_time_segundos != float('inf') and max_time_segundos > max_time_general_segundos):
                          max_time_general_segundos = max_time_segundos
                          max_time_from_general = max_time_from
                          max_time_to_general = max_time_to
                 elif max_time_general_segundos == 0: # If this is the first route and max_time is inf
                      max_time_general_segundos = float('inf')
                      max_time_from_general = max_time_from
                      max_time_to_general = max_time_to


            # Calcular tiempos acumulados
            tiempo_acumulado_segundos = 0
            tiempos_acumulados_hhmmss = []
            # Convertir hora_inicio_dt a un objeto datetime completo (con fecha arbitraria) para poder sumarle timedeltas
            hora_inicio_completa = datetime.combine(datetime.today(), hora_inicio_dt)

            for i in range(len(tiempos_segundos)):
                tiempo_acumulado_segundos += tiempos_segundos[i]
                tiempo_actual_dt = hora_inicio_completa + timedelta(seconds=tiempo_acumulado_segundos)
                tiempos_acumulados_hhmmss.append(tiempo_actual_dt.strftime('%H:%M:%S'))


            # Exportar filas detalladas del recorrido
            for idx, codigo in enumerate(codigos):
                # Obtener la distancia y tiempo al siguiente punto (o vacío if es el último punto en la lista)
                distancia_val = distancias[idx] if idx < len(distancias) else ''
                tiempo_val_segundos = tiempos_segundos[idx] if idx < len(tiempos_segundos) else ''
                tiempo_val_hhmmss = str(timedelta(seconds=tiempo_val_segundos)) if tiempo_val_segundos != '' and tiempo_val_segundos != float('inf') else ''
                tiempo_acumulado_val_hhmmss = tiempos_acumulados_hhmmss[idx-1] if idx > 0 and idx <= len(tiempos_acumulados_hhmmss) else hora_inicio_dt.strftime('%H:%M') if idx == 0 else '' # El primer punto es la hora de inicio, los siguientes son acumulados

                export_rows.append({
                    'MET': met,
                    'Ruta': ruta,
                    'Secuencia': idx+1,
                    'Tipo': tipos[idx],
                    'Codigo': codigo,
                    'Longitud': puntos[idx][0],
                    'Latitud': puntos[idx][1],
                    'Nombre': nombres[idx],
                    'Distancia_al_siguiente_km': distancia_val,
                    'Tiempo_al_siguiente_segundos': tiempo_val_segundos,
                    'Tiempo_al_siguiente_HHMMSS': tiempo_val_hhmmss,
                    'Tiempo_acumulado_HHMM': tiempo_acumulado_val_hhmmss,
                    'Recorrido': ' -> '.join([str(x) for x in recorrido_codigos]),
                    'Distancia_total_ruta_km': distancia_total_ruta, # Distancia total de la ruta
                    'Tiempo_total_ruta_segundos': tiempo_total_ruta_segundos, # Tiempo total de la ruta
                    'Tiempo_total_ruta_HHMMSS': str(timedelta(seconds=tiempo_total_ruta_segundos)) if tiempo_total_ruta_segundos != float('inf') else float('inf')
                })


    print("Finalizado el procesamiento de rutas por secuencia. Preparando para la exportación a Excel...")
    # Exportar a Excel con formato profesional
    df_export = pd.DataFrame(export_rows)
    df_resumen = pd.DataFrame(resumen_rutas)
    resumen_general = pd.DataFrame([{
        'Total_clientes_procesados': total_clientes_general,
        'Distancia_total_rutas_km': distancia_total_general,
        'Distancia_promedio_cliente_general_km': distancia_total_general / total_clientes_general if total_clientes_general > 0 else 0,
        'Distancia_maxima_en_ruta_km': max_dist_general,
        'De_Distancia_General': max_from_general,
        'A_Distancia_General': max_to_general,
        'Tiempo_total_rutas_segundos': tiempo_total_general_segundos,
        'Tiempo_total_rutas_HHMMSS': str(timedelta(seconds=tiempo_total_general_segundos)) if tiempo_total_general_segundos != float('inf') else float('inf'),
        'Tiempo_promedio_cliente_general_segundos': tiempo_total_general_segundos / total_clientes_general if total_clientes_general > 0 else 0,
        'Tiempo_promedio_cliente_general_HHMMSS': str(timedelta(seconds=tiempo_total_general_segundos / total_clientes_general)) if total_clientes_general > 0 and tiempo_total_general_segundos != float('inf') else 0,
        'Tiempo_maximo_en_ruta_segundos': max_time_general_segundos,
        'Tiempo_maximo_en_ruta_HHMMSS': str(timedelta(seconds=max_time_general_segundos)) if max_time_general_segundos != float('inf') else float('inf'),
        'De_Tiempo_General': max_time_from_general,
        'A_Tiempo_General': max_time_to_general
    }])


    excel_path = os.path.join(output_dir, f'recorridos_secuencia_todos_{fecha_hora}.xlsx')

    try:
        print(f"Exportando a Excel: {excel_path}...")
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            df_export.to_excel(writer, sheet_name='Recorridos', index=False)
            df_resumen.to_excel(writer, sheet_name='Resumen por ruta', index=False)
            resumen_general.to_excel(writer, sheet_name='Resumen general', index=False)

        print("Aplicando formato profesional al archivo Excel...")
        # Aplicar formato profesional
        wb = openpyxl.load_workbook(excel_path)
        header_font = Font(bold=True, color='FFFFFF', size=12)
        header_fill = PatternFill('solid', fgColor='4F81BD')
        cell_fill = PatternFill('solid', fgColor='DDEBF7')
        border = Border(left=Side(style='thin', color='4F81BD'), right=Side(style='thin', color='4F81BD'), top=Side(style='thin', color='4F81BD'), bottom=Side(style='thin', color='4F81BD'))
        align = Alignment(horizontal='center', vertical='center', wrap_text=True)

        for sheet_name in wb.sheetnames:
            ws = wb[sheet_name]
            # Aplicar formato a la fila de encabezado (fila 1)
            for col_idx, cell in enumerate(ws[1], 1):
                cell.font = header_font
                cell.fill = header_fill
                cell.alignment = align
                cell.border = border
                # Añadir emojis a los encabezados, verificando que cell.value no sea None
                if cell.value:
                    if 'Ruta' in str(cell.value): cell.value = '🛣️ ' + str(cell.value)
                    elif 'Cliente' in str(cell.value): cell.value = '👨‍💼 ' + str(cell.value)
                    elif 'Distancia' in str(cell.value): cell.value = '📏 ' + str(cell.value)
                    elif 'Tiempo' in str(cell.value): cell.value = '⏳ ' + str(cell.value) # Añadir emoji para tiempo
                    elif 'Secuencia' in str(cell.value): cell.value = '🔢 ' + str(cell.value)
                    elif 'MET' in str(cell.value): cell.value = '🏠 ' + str(cell.value)
                    elif 'Nombre' in str(cell.value): cell.value = '📚 ' + str(cell.value)
                    elif 'De' in str(cell.value): cell.value = '🔴 ' + str(cell.value) # Modificado para incluir "De_Distancia", "De_Tiempo", etc.
                    elif 'A' in str(cell.value): cell.value = '🟢 ' + str(cell.value) # Modificado para incluir "A_Distancia", "A_Tiempo", etc.
                    elif 'Codigo' in str(cell.value): cell.value = '🆔 ' + str(cell.value) # Add emoji for Codigo
                    elif 'Longitud' in str(cell.value) or 'Latitud' in str(cell.value): cell.value = '🌍 ' + str(cell.value) # Add emoji for coordinates


            # Aplicar formato a las filas de datos (a partir de la fila 2)
            for row in ws.iter_rows(min_row=2):
                for cell in row:
                    cell.fill = cell_fill
                    cell.alignment = align
                    cell.border = border

        # Ajustar anchos de columna
        for col in ws.columns:
            max_length = 0
            col_letter = get_column_letter(col[0].column)
            for cell in col:
                try:
                    if cell.value:
                        # Aumentar el ancho para emojis o caracteres especiales
                        max_length = max(max_length, len(str(cell.value)) + str(cell.value).count(' '))
                except:
                    pass
            # Establecer un ancho mínimo y máximo razonable
            ws.column_dimensions[col_letter].width = max(15, min(max_length + 3, 60)) # Ajustado ancho min/max y padding

        # Aplicar un color de relleno diferente para las filas de datos en las hojas de resumen
        if sheet_name in ['Resumen general', 'Resumen por ruta']:
            for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
                for cell in row:
                    cell.fill = PatternFill('solid', fgColor='FFD966')


        wb.save(excel_path)
        print(f'Excel grupal exportado y formateado a: {excel_path}')

        # 7. Manejar la descarga del archivo de salida
        # Descargar el archivo Excel generado
        try:
            if os.path.exists(excel_path):
                print(f"Iniciando descarga del archivo: {excel_path}...")
                files.download(excel_path)
                print("Descarga iniciada. Revisa tus descargas locales.")
            else:
                print(f"Error: El archivo de salida '{excel_path}' no fue encontrado después de la generación. Es posible que haya habido un error en un paso anterior.")
        except Exception as e:
            print(f"Ocurrió un error durante la descarga del archivo: {e}")


    except Exception as e:
        print(f"Ocurrió un error durante la exportación o el formato de Excel: {e}")
        print(f"El archivo no fue exportado a: {excel_path}")


print("Ejecución del script finalizada.")

Por favor, selecciona el archivo Excel de entrada.


Saving recorridos_optimos_60%vol_190226.xlsx to recorridos_optimos_60%vol_190226 (1).xlsx
Archivo 'recorridos_optimos_60%vol_190226 (1).xlsx' cargado exitosamente.
Intentando cargar datos desde recorridos_optimos_60%vol_190226 (1).xlsx...
Datos cargados exitosamente.
Dimensiones de df_puntos_secuencia: (13792, 8)
Asignando parámetros desde la configuración manual...
Procesando todas las rutas disponibles.
Procesando todos los clientes en las rutas seleccionadas.
Procesando con MET(s): ['MET CDMX2', 'MET Ceylán']
Procesando TODAS las rutas disponibles.
Hora de inicio de rutas: 07:00
Iniciando el procesamiento de rutas por secuencia...
Asegurando que el directorio de salida '/tmp/outputs' exista...
Procesando MET: MET CDMX2, Ruta: 1
Calculando distancias y tiempos consecutivos para 221 puntos...
Procesando MET: MET CDMX2, Ruta: 12
Calculando distancias y tiempos consecutivos para 312 puntos...
Procesando MET: MET CDMX2, Ruta: 14
Calculando distancias y tiempos consecutivos para 195 punto

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descarga iniciada. Revisa tus descargas locales.
Ejecución del script finalizada.
